# Assignment 4.1: Retrieval-Augmented Question Answering Using LangChain

## Objective
This notebook builds an end-to-end Retrieval-Augmented Generation (RAG) pipeline over the TED Talks dataset:

1. Load and structure the TED dataset using LangChain `Document`.
2. Process the full selected 1,000-record corpus rather than only the first 50 records.
3. Split the documents into overlapping chunks.
4. Generate semantic embeddings with Hugging Face `all-MiniLM-L6-v2`.
5. Store embeddings in a FAISS vector database.
6. Configure a semantic retriever and inspect retrieved evidence.
7. Use `google/flan-t5-small` with the correct **text2text-generation** pipeline.
8. Build a LangChain `RetrievalQA` chain.
9. Test open-domain questions related to climate change and education and display the supporting retrieved documents.

> **Important correction from the earlier version:** FLAN-T5 is a sequence-to-sequence model, so it must be used with `text2text-generation`, not `text-generation`. The earlier notebook also indexed only `documents[:50]`, which excluded most of the loaded corpus.


In [18]:
# Install/import dependencies if needed
# Uncomment the following line in a fresh environment:
# %pip install -U datasets langchain-core langchain-text-splitters langchain-community langchain-huggingface langchain-classic sentence-transformers faiss-cpu transformers torch

from datasets import load_dataset
from langchain_core.documents import Document

# Load the selected TED Talks corpus.
# We intentionally use all 1,000 records selected here; we do NOT reduce them to documents[:50].
dataset = load_dataset(
    "IWSLT/ted_talks_iwslt",
    language_pair=("nl", "en"),
    year="2016",
    split="train[:1000]",
    trust_remote_code=True
)

print("Number of dataset records:", len(dataset))
print("Dataset columns:", dataset.column_names)


Number of dataset records: 1000
Dataset columns: ['translation']


In [19]:
# Convert every selected dataset record into a LangChain Document.
# The code is defensive because dataset schemas can vary slightly across versions.

documents = []

for idx, item in enumerate(dataset):
    translation = item.get("translation", {})
    text = translation.get("en", "") if isinstance(translation, dict) else ""

    if text and text.strip():
        metadata = {
            "record_id": idx,
            "speaker": item.get("speaker", "Unknown"),
            "talk_id": item.get("talk_id", item.get("id", idx)),
            "translation_pair": "nl_en_2016",
        }

        documents.append(
            Document(
                page_content=text.strip(),
                metadata=metadata
            )
        )

print("LangChain documents created:", len(documents))
print("\nExample document:")
print(documents[0])


LangChain documents created: 1000

Example document:
page_content='We don't have to live in a world where 99 percent of rapists get away with it, says TED Fellow Jessica Ladd. With Callisto, a new platform for college students to confidentially report sexual assault, Ladd is helping survivors get the support and justice they deserve while respecting their privacy concerns. "We can create a world where there's a real deterrent to violating the rights of another human being," she says.' metadata={'record_id': 0, 'speaker': 'Unknown', 'talk_id': 0, 'translation_pair': 'nl_en_2016'}


## Document processing and chunking

A chunk size of 500 characters with 50-character overlap provides enough context for semantic retrieval while keeping the input manageable for the QA model.


In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

# IMPORTANT: process the complete selected corpus, not documents[:50].
docs = text_splitter.split_documents(documents)

print("Original LangChain documents:", len(documents))
print("Total chunks indexed:", len(docs))

print("\nSample chunk:")
print(docs[0].page_content)
print("\nMetadata:", docs[0].metadata)


Original LangChain documents: 1000
Total chunks indexed: 1053

Sample chunk:
We don't have to live in a world where 99 percent of rapists get away with it, says TED Fellow Jessica Ladd. With Callisto, a new platform for college students to confidentially report sexual assault, Ladd is helping survivors get the support and justice they deserve while respecting their privacy concerns. "We can create a world where there's a real deterrent to violating the rights of another human being," she says.

Metadata: {'record_id': 0, 'speaker': 'Unknown', 'talk_id': 0, 'translation_pair': 'nl_en_2016'}


## Embeddings and FAISS vector store

Each text chunk is converted into a dense vector using the Hugging Face Sentence Transformer model `all-MiniLM-L6-v2`. FAISS then indexes these vectors so that semantically similar content can be retrieved efficiently.

Using `langchain_huggingface` avoids the deprecated `HuggingFaceEmbeddings` import from `langchain_community`.


In [22]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embeddings
)

print("FAISS vector store created successfully.")
print("Indexed vectors:", vectorstore.index.ntotal)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS vector store created successfully.
Indexed vectors: 1053


## Semantic retrieval

The retriever returns the most semantically similar chunks rather than relying on exact keyword matching. The retrieved documents are displayed below so that the retrieval stage can be validated before generating an answer.


In [24]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

def show_retrieval_results(question):
    results = retriever.invoke(question)

    print("QUESTION:", question)
    print("=" * 80)

    for i, doc in enumerate(results, start=1):
        print(f"\nResult {i}")
        print("-" * 80)
        print("Metadata:", doc.metadata)
        print("Content:", doc.page_content)

    return results

# Example semantic retrieval validation
retrieved_docs = show_retrieval_results(
    "What are some ways people can address climate change?"
)


QUESTION: What are some ways people can address climate change?

Result 1
--------------------------------------------------------------------------------
Metadata: {'record_id': 97, 'speaker': 'Unknown', 'talk_id': 97, 'translation_pair': 'nl_en_2016'}
Content: Alice Bows-Larkin: Climate change is happening. Here's how we adapt

Result 2
--------------------------------------------------------------------------------
Metadata: {'record_id': 106, 'speaker': 'Unknown', 'talk_id': 106, 'translation_pair': 'nl_en_2016'}
Content: Climate change is unfair. While rich countries can fight against rising oceans and dying farm fields, poor people around the world are already having their lives upended -- and their human rights threatened -- by killer storms, starvation and the loss of their own lands. Mary Robinson asks us to join the movement for worldwide climate justice.

Result 3
--------------------------------------------------------------------------------
Metadata: {'record_id': 522, 's

## FLAN-T5 question-answering model

`google/flan-t5-small` is a sequence-to-sequence model.


In [25]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_core.language_models.llms import LLM
from typing import Optional, List

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


class FlanT5LLM(LLM):

    @property
    def _llm_type(self) -> str:
        return "flan-t5"

    def _call(
        self,
        prompt: str,
        stop: Optional[List[str]] = None,
        run_manager=None,
        **kwargs
    ) -> str:

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512
        )

        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )

        return tokenizer.decode(
            outputs[0],
            skip_special_tokens=True
        )


llm = FlanT5LLM()

# Sanity test
test_prompt = "Explain artificial intelligence in simple terms."

print("Model sanity check:")
print(llm.invoke(test_prompt))

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model sanity check:
Artificial intelligence is a technology that can be used to measure the size of a machine.


## RetrievalQA pipeline

The prompt explicitly instructs the model to use only the retrieved TED context. This is important for RAG because the generated answer should be grounded in retrieved evidence rather than being a free-form answer.

The chain also returns the source documents so that the answer can be checked against the retrieved evidence.


In [26]:
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

prompt_template = """Use the following TED Talks context to answer the question.

If the answer cannot be found in the context, say:
I don't know based on the retrieved TED Talks.

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}
)

print("RetrievalQA chain created successfully.")


RetrievalQA chain created successfully.


## Required QA demonstrations

The following questions are intentionally open-domain questions on **climate change** and **education**, as required by the assignment. Each response is accompanied by the retrieved source chunks so that the answer can be checked for grounding.


In [27]:
def ask_question(question):
    result = qa_chain.invoke({"query": question})

    print("\nQUESTION:")
    print(question)

    print("\nANSWER:")
    print(result["result"])

    print("\nRETRIEVED EVIDENCE:")
    for i, doc in enumerate(result["source_documents"], start=1):
        print(f"\nSource {i} | metadata={doc.metadata}")
        print(doc.page_content)

    return result

climate_question = (
    "What challenges or actions related to climate change are discussed "
    "in the retrieved TED Talks?"
)

education_question = (
    "What ideas or approaches to education and learning are discussed "
    "in the retrieved TED Talks?"
)

climate_result = ask_question(climate_question)
education_result = ask_question(education_question)



QUESTION:
What challenges or actions related to climate change are discussed in the retrieved TED Talks?

ANSWER:
(iii).

RETRIEVED EVIDENCE:

Source 1 | metadata={'record_id': 97, 'speaker': 'Unknown', 'talk_id': 97, 'translation_pair': 'nl_en_2016'}
Alice Bows-Larkin: Climate change is happening. Here's how we adapt

Source 2 | metadata={'record_id': 62, 'speaker': 'Unknown', 'talk_id': 62, 'translation_pair': 'nl_en_2016'}
For the people of Kiribati, climate change isn't something to be debated, denied or legislated against -- it's an everyday reality. The low-lying Pacific island nation may soon be underwater, thanks to rising sea levels. In a personal conversation with TED Curator Chris Anderson, Kiribati President Anote Tong discusses his country's present climate catastrophe and its imperiled future. "In order to deal with climate change, there's got to be sacrifice. There's got to be commitment," he says.

Source 3 | metadata={'record_id': 665, 'speaker': 'Unknown', 'talk_id':